# CHIMESim — Tutorial

CHIMESim is a Couzin-style agent-based simulation with three social interaction zones (repulsion, orientation, attraction) and support for multiple agent types.

Install once if needed:
```bash
pip install -e .
```

## 1 — Launch the default simulation

Runs a single group by default, or restores the previous GUI session.  
The window opens interactively; **the kernel is blocked until you close it.**
The simulation, controls, information and each agent type use separate windows.
Set `width` and `height` in `chimesim.run()` for the fixed arena size;
`window_width` and `window_height` only set the initial display size.
Resize the display freely; change the arena through **Apply size & reset** in Controls.
Use the **Windows** menu to restore hidden panels.


In [ ]:
import chimesim

chimesim.run()

## 2 — Custom agent types

Pass a list of `AgentType` objects to `run()` to define your own scenario.  
Each type has:

| Parameter | What it controls |
|---|---|
| `color` | Triangle fill colour (any Tk colour string) |
| `size` | Body length in pixels |
| `speed_scale` | Multiplier on the global Speed slider |
| `max_turn` | Maximum heading change per step (radians) |
| `noise` | Gaussian heading-noise standard deviation (radians) |
| `hunts` | Type names this type chases and catches |
| `flees` | Type names this type runs away from |

Each agent type has its own **N** slider. Types without adversarial relationships interact socially.

In [ ]:
# --- Example A: classic predator–prey ---
# An explicit predator–prey configuration you can tweak.
import chimesim, math

prey = chimesim.AgentType(
    "prey",
    color="#56b1fc",
    size=15,
    speed_scale=1.0,
    max_turn=math.pi * 0.2,
    flees=("predator",),
)

predator = chimesim.AgentType(
    "predator",
    color="#fc6056",
    size=20,
    speed_scale=1.5,       # predators move 50% faster than the Speed slider value
    max_turn=math.pi * 0.15,
    noise=0.3,             # Gaussian heading-noise standard deviation (radians)
    hunts=("prey",),
)

chimesim.run(agent_types=[prey, predator])

In [ ]:
# --- Example B: fast shoal vs slow shoal ---
# Two groups with the same social rules, different speeds.
# Neither hunts the other — they just coexist.
import chimesim

slow_shoal = chimesim.AgentType("slow", color="#56b1fc", size=14, speed_scale=0.5)
fast_shoal = chimesim.AgentType("fast", color="#f4a030", size=14, speed_scale=2.0)

chimesim.run(agent_types=[slow_shoal, fast_shoal])

In [ ]:
# --- Example C: two socially interacting shoals ---
# Same speed, different colours.  Useful for studying group mixing.
import chimesim

shoal_a = chimesim.AgentType("shoal_a", color="#56b1fc", size=14)
shoal_b = chimesim.AgentType("shoal_b", color="#a060f0", size=14)

chimesim.run(agent_types=[shoal_a, shoal_b])

In [ ]:
# --- Example D: timid vs bold ---
# Both types flee predators, but the bold group turns more slowly
# and has a wider attraction zone (set via sliders at runtime).
import chimesim, math

timid = chimesim.AgentType(
    "timid",
    color="#56b1fc",
    size=13,
    max_turn=math.pi * 0.3,   # turns fast — more reactive
    speed_scale=1.2,
    flees=("predator",),
)

bold = chimesim.AgentType(
    "bold",
    color="#40c090",
    size=16,
    max_turn=math.pi * 0.1,   # turns slowly — less reactive
    speed_scale=1.0,
    flees=("predator",),
)

predator = chimesim.AgentType(
    "predator",
    color="#fc6056",
    size=20,
    speed_scale=1.5,
    hunts=("timid", "bold"),  # hunts both
)

chimesim.run(agent_types=[timid, bold, predator])

## 3 — Headless simulation

For most experiments use `chimesim.simulate(n_steps=500, n_reps=5, seed=42)`.
It handles burn-in, repeats, metrics and optional trajectories without Tkinter.
Pass a dictionary or JSON file as `settings`; returned `results["settings"]`
contains the resolved configuration for reuse. See the README for examples.

The lower-level examples below show how to step a model manually. Zone radii
and detection range belong to `AgentType`; `model.step()` accepts
`catch_dist`, `desired_speed` and `fov_deg`. A model without an attached
`Environment` uses periodic boundaries. Distances are pixels and time is steps.


In [ ]:
import chimesim

N_STEPS = 500
N_PREY  = 100
N_PREDS = 5

# Zone radii and flee_dist are now per-type attributes on AgentType.
# Set them before or after creating the model — they're read live each step.
prey      = chimesim.AgentType("prey",     r_r=10, r_o=30, r_a=75,  flee_dist=100,
                             flees=("predator",), n_default=N_PREY)
predator  = chimesim.AgentType("predator", r_r=10, r_o=50, r_a=150, flee_dist=100,
                             speed_scale=1.5, hunts=("prey",), n_default=N_PREDS)

model = chimesim.Model(agent_types=[prey, predator], seed=42)
model.init_agents("prey",     N_PREY)
model.init_agents("predator", N_PREDS)

prey_counts = [len(model.groups["prey"])]

for _ in range(N_STEPS):
    model.step(catch_dist=40, desired_speed=3)
    prey_counts.append(len(model.groups["prey"]))

print(f"Start: {prey_counts[0]} prey")
print(f"End ({N_STEPS} steps): {prey_counts[-1]} prey")
print(f"Survival: {prey_counts[-1] / N_PREY:.0%}")

In [ ]:
# Plot survival over time (requires matplotlib)
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 3))
plt.plot(prey_counts, color="#56b1fc")
plt.xlabel("Step")
plt.ylabel("Prey alive")
plt.tight_layout()
plt.show()

In [ ]:
import chimesim

N_STEPS = 300

scenarios = {
    "small ZOA (r_a=40)":  dict(r_r=10, r_o=25, r_a=40),
    "medium ZOA (r_a=75)": dict(r_r=10, r_o=30, r_a=75),
    "large ZOA (r_a=150)": dict(r_r=10, r_o=40, r_a=150),
}

for label, zones in scenarios.items():
    prey     = chimesim.AgentType("prey",     **zones, flee_dist=100,
                                flees=("predator",), n_default=100)
    predator = chimesim.AgentType("predator", r_r=10, r_o=50, r_a=150,
                                speed_scale=1.5, hunts=("prey",), n_default=5)
    m = chimesim.Model(agent_types=[prey, predator], seed=42)
    m.init_agents("prey",     100)
    m.init_agents("predator", 5)
    for _ in range(N_STEPS):
        m.step(catch_dist=40, desired_speed=3)
    print(f"{label}: {len(m.groups['prey'])}/100 prey survived")